In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [ ]:
df_test = pd.read_csv('test.csv')
df_train = pd.read_csv('train.csv')

In [ ]:
df_train.head()

In [ ]:
df_test.head()

In [ ]:
df_test.info()

In [ ]:
df_train.info()

In [ ]:
# Target distribution
plt.figure(figsize=(6,4))
sns.histplot(df_train["Units_Sold"], bins=20, kde=True)
plt.title("Units Sold Distribution")
plt.show()

In [ ]:
sns.scatterplot(x="Discount_Percentage", y="Units_Sold", data=df_train)
plt.title("Discount vs Sales")
plt.show()

In [ ]:
sns.scatterplot(x="Battery_Capacity_kWh", y="Units_Sold", data=df_train)
plt.title("Battery Capacity vs Sales")
plt.show()

In [ ]:

# Region impact
plt.figure(figsize=(8,5))
sns.boxplot(x="Region", y="Units_Sold", data=df_train)
plt.xticks(rotation=45)
plt.show()

In [ ]:
def preprocess(df):
    df = df.copy()
    
    # Convert Date
    df["Date"] = pd.to_datetime(df["Date"])
    
    # Extract time features
    df["Year"] = df["Date"].dt.year
    df["Month"] = df["Date"].dt.month
    
    # Drop unnecessary columns
    df = df.drop(columns=["Date", "Model"])
    
    return df

train = preprocess(df_train)
test = preprocess(df_test)

In [ ]:
combined = pd.concat([df_train.drop("Units_Sold", axis=1), test])

combined = pd.get_dummies(combined, drop_first=True)


In [ ]:
X = combined.iloc[:len(train)]
X_test = combined.iloc[len(train):]

# print(X)
y = train["Units_Sold"]

In [ ]:
X_train, X_val, y_train, y_val = train_test_split( X, y, test_size=0.2, random_state=42)

In [ ]:
model = RandomForestRegressor(
    n_estimators=200,
    max_depth=None,
    random_state=42
)

model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_val)

mae = mean_absolute_error(y_val, y_pred)
rmse = np.sqrt(mean_squared_error(y_val, y_pred))

print("MAE:", mae)
print("RMSE:", rmse)

In [ ]:
importances = pd.Series(model.feature_importances_, index=X.columns)
importances.sort_values().tail(15).plot(kind="barh", figsize=(8,5))
plt.title("Top Feature Importance")
plt.show()

In [ ]:
model.fit(X, y)

In [ ]:
test_predictions = model.predict(X_test)

test["Predicted_Units_Sold"] = test_predictions

print(test.head())

In [ ]:
df_train["Quarter"] = df_train["Date"].dt.quarter
df_train["DayOfWeek"] = df_train["Date"].dt.dayofweek

In [ ]:
import pickle
with open("ev_sales_model.pkl", "wb") as f:
    pickle.dump(model, f)


test.to_csv("ev_sales_predictions.csv", index=False)